#### Imports and setup

This cell imports helper functions and creates:
- a gold_run_id value
- a run date string
- a run timestamp string

These are used for tracking and snapshot publishing

In [ ]:
%run "../notebooks/helper_functions"

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [ ]:
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().stftime("%Y-%m-%d")

print(f"Current Gold Run ID: {gold_run_id}")
print(f"Run timestamp Folder: {run_ts_str}")

#### Read changed Silver rows only

This cell reads the full Silver current-state tables but filters only the rows that changed since the last Gold run

This is the starting poing for Gold incremental processing.

In [ ]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("Last Processed Silver Timestamp for Gold=", last_gold_ts)

silver_orders_current = spark.read.table("novacart_catalog.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_catalog.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_catalog.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silvers_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silvers_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silvers_payments_current.filter(F.col("updated_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders = {changed_orders_count}")
print(f"Number of changed products = {changed_products_count}")
print(f"Number of changed payments = {changed_payments_count}")

#### Find impacted order IDs
Gold is built at order grain, so if anything changes in orders, products or payments, we identify which order_id values are impacted.
Only those order IDs are rebuilt in Gold.

In [ ]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"),
          "inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_orders)
    .distinct()
)

print(f"impacted order ids = {impacted_order_ids.count()}")
display(impacted_order_ids.orderBy("order_id"))

#### Build Gold delta for impacted orders
This cell joins the impacted orders with the current Silver products and payments tables, derives business columns, and builds the **Gold delta** that will be merged into the Gold current-state table.

In [ ]:
impacted_order = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_order.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        F.col("o.order_id") == F.col("py.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.products_name"),
        F.col("p.category"),
        F.col("p.price").alias("products_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.updated_at").cast("timestamp"),
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0, "Unpaid")
        .when(F.col("payment_completion_ratio") == 1, "Paid")
        .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
        .when(F.col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

print("Gold delta rows =",gold_delta.count())
display(gold_delta)

#### Merge Gold current-state table
If Gold delta contains rows, this cell merges them into gold_schema.orders_information. If there are no impacted rows, nothing is merged

In [ ]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novacart_catalog.gold_schema.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")

In [ ]:
%sql
select * from novacart_catalog.gold_schema.orders_information;

#### Maintain Gold SCD Type 2 history
This cell updates the SCD2 history table.

If a current Gold row changes, the old version is closed(is_current = false) and a new current version is inserted.

In [ ]:
if not spark.catalog.tableExists("novacart_catalog.gold_schema.orders_information_scd2"):
    spark.sql("""
    create tables novacart_catalog.gold_schema.orders_information_scd2
    using delta as
    select *, cast(null as timestamp) as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
    from novacart_catalog.gold_schema.orders_information
    where 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
    merge into novacart_catalog.gold_schema.orders_information_scd2 t
    using gold_delta_view s
    on t.order_id = s.order_id and t.is_current = true
    when matched and (
        not(t.order_status <=> s.order_status) or 
        not(t.order_amount <=> s.order_amount) or 
        not(t.paid_amount <=> s.paid_amount) or 
        not(t.payment_id <=> s.payment_id) or 
        not(t.category <=> s.category) or 
        not(t.product_name <=> s.product_name) or 
        not(t.products_price <=> s.product_price) )
    then update set 
        is_current = false,
        valid_to_ts = s.gold_update_ts
    """)

    spark.sql("""
    insert into novacart_catalog.gold_schema.orders_information_scd2
    select s.*,
           s.gold_update_ts as valid_from_ts,
           cast(null as timestamp) as valid_to_ts,
           true as is_current
    from gold_delta_view s 
    left join novacart_catalog.gold_schema.orders_information_scd2 t 
    on s.order_id = t.order_id and t.is_current = true 
    where t.order_id is null or (
        not(t.order_status <=> s.order_status) or 
        not(t.order_amount <=> s.order_amount) or 
        not(t.paid_amount <=> s.paid_amount) or
        not(t.payment_id <=> s.payment_id) or
        not(t.category <=> s.category) or
        not(t.product_name <=> s.product_name) or
        not(t.products_price <=> s.products_price)
    )
    """)

#### Update category-level Gold aggregation

This cell recalculates category-level business metrics only for categories impacted in the current run, then merges them into the category performance Gold table

In [ ]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("novacart_catalog.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Gross Merchandise Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg(F.col("payment_completion_ratio")).alias("Average Payment Completion Ratio"),
            ( F.sum(
                F.when(F.col("payment_status") == "FAILED", 1).otherwise(0)
            )/ F.count("*").alias("Payment_Failure_Rate"))
        )
    )
    upsert_to_gold(category_perf_delta, "novacart_catalog.gold_schema.category_performance", "category")
else:
    print("No new rows to insert in gold table")

In [ ]:
%sql
select * from novacart_catalog.gold_schema.category_performance;

#### Publish Gold snapshots to Volumne
This cell writes two kinds of Gold outputs to a Databricks Volume
- **latest snapshot** - overwritten every successful run
- **timestamped historical snapshot** - a new folder for each successful run

This is useful for audit, rollback, and teaching demos. 

In [ ]:
spark.sql("create volumne if not exists novacart_catalog.gold_schema.gold_snapshots_vol")

In [ ]:
latest_orders_path = (
    "/Volumes/novacart_catalog/gold_schema/gold_snaphots_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumnes/novacart_catalog/gold_schema/gold_snaphots_vol/gold_latest/category_performance"
)

historical_orders_path = f"/Volumes/novacart_catalog/gold_schema/gold_snaphots_vol/gold_snpshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path = f"/Volumes/novacart_catalog/gold_schema/gold_snaphots_vol/gold_snpshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

spark.read.table("novacart_catalog.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("novacart_catalog.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print(f"Latest Orders Path: {latest_orders_path}")
print(f"Latest Category Path: {latest_category_path}")
print(f"Historical Orders Path: {historical_orders_path}")
print(f"Historical Category Path: {historical_category_path}")

#### Update Gold Control Table
This final cell updates the Gold COntrol table using the latest Silver processing metadata and displays the control table for validation

In [ ]:
latest_silver_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

upsert_gold_control("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())

display(spark.table("novacart_catalog.gold_schema.processing_control"))